In [ ]:
!pip install pycolmap
!pip install git+https://github.com/cvg/LightGlue.git

!apt-get install -y colmap

In [ ]:
import pandas as pd
import os
import shutil
import numpy as np
from scipy.spatial.transform import Rotation as R
import cv2
import torch
import pycolmap
import array
import subprocess
from lightglue import ALIKED
from lightglue import LightGlue

In [ ]:
data_path = "/kaggle/input/image-matching-challenge-2025"
test_path = os.path.join(data_path, "test")

# Загрузка sample_submission.csv
submission = pd.read_csv(os.path.join(data_path, "sample_submission.csv"))

In [ ]:
# Функция для разделения изображений на группы по размерам (для ETs)
def group_images_by_size(dataset, scene, group):
    image_sizes = {}
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            img = cv2.imread(image_path)
            if img is not None:
                size = img.shape[:2]
                size_key = f"{size[0]}x{size[1]}"
                if size_key not in image_sizes:
                    image_sizes[size_key] = []
                image_sizes[size_key].append((idx, row))
    return image_sizes

In [ ]:
def process_group(dataset, scene, group, indices):
    project_dir = f"/kaggle/working/project_{dataset}_{scene}_{indices[0]}"
    os.makedirs(project_dir, exist_ok=True)
    db_path = os.path.join(project_dir, "database.db")
    
    # Удаление старой базы данных, если существует (как в main.ipynb)
    if os.path.exists(db_path):
        os.remove(db_path)
    
    # Копирование изображений в рабочую директорию
    image_names = []
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            shutil.copy(image_path, project_dir)
            image_names.append(row['image'])
    
    # Пропуск, если изображений меньше 2 (как в main.ipynb)
    if len(image_names) < 2:
        print(f"Skipping group with insufficient images: {len(image_names)}")
        shutil.rmtree(project_dir)
        return
    
    # Инициализация устройств и моделей
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    aliked = ALIKED().to(device)
    matcher = LightGlue(features='aliked').to(device)
    
    # Создание директории для признаков и файла параметров камеры
    features_dir = os.path.join(project_dir, "features")
    os.makedirs(features_dir, exist_ok=True)
    camera_params_file = os.path.join(project_dir, "camera_params.txt")
    
    # Извлечение признаков с помощью ALIKED и сохранение в текстовые файлы
    with open(camera_params_file, 'w') as f:
        for image_name in image_names:
            img = cv2.imread(os.path.join(project_dir, image_name))
            h, w = img.shape[:2]
            focal = 1.2 * max(w, h)
            cx, cy = w / 2, h / 2
            f.write(f"{image_name} {focal},{cx},{cy}\n")
            
            # Извлечение признаков с ALIKED
            img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().to(device) / 255.0
            feats = aliked.extract(img_tensor)
            keypoints = feats['keypoints'].cpu().numpy()  # Убираем индекс [0]
            descriptors = feats['descriptors'].cpu().numpy()  # Убираем индекс [0]
            
            # Сохранение признаков в текстовый файл в формате COLMAP
            feature_file = os.path.join(features_dir, f"{image_name}.txt")
            with open(feature_file, 'w') as ff:
                ff.write(f"{len(keypoints)} 128\n")  # 128 - размер дескриптора ALIKED
                for kp, desc in zip(keypoints, descriptors):
                    ff.write(f"{kp[0]} {kp[1]} 1 0")  # x, y, scale, orientation
                    for d in desc:
                        ff.write(f" {d}")
                    ff.write("\n")
    
    # Импорт признаков в базу данных COLMAP
    try:
        result = subprocess.run([
            "colmap", "feature_importer",
            "--database_path", db_path,
            "--image_path", project_dir,
            "--import_path", features_dir,
            "--ImageReader.camera_model", "SIMPLE_PINHOLE",
            "--ImageReader.camera_params", camera_params_file
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print("Feature importer output:", result.stdout)
        print("Feature importer errors:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error in feature_importer: {e.stderr}")
        shutil.rmtree(project_dir)
        return
    
    # Создание директории для совпадений и файла пар
    matches_dir = os.path.join(project_dir, "matches")
    os.makedirs(matches_dir, exist_ok=True)
    pairs_file = os.path.join(project_dir, "pairs.txt")
    
    # Сопоставление признаков с помощью LightGlue
    with open(pairs_file, 'w') as pf:
        for i in range(len(image_names)):
            for j in range(i + 1, len(image_names)):
                image_name1, image_name2 = image_names[i], image_names[j]
                
                # Чтение дескрипторов из файлов
                desc1 = np.loadtxt(os.path.join(features_dir, f"{image_name1}.txt"), 
                                 skiprows=1, usecols=range(4, 132)).astype(np.float32)
                desc2 = np.loadtxt(os.path.join(features_dir, f"{image_name2}.txt"), 
                                 skiprows=1, usecols=range(4, 132)).astype(np.float32)
                
                # Сопоставление с LightGlue
                matches = matcher.match(
                    torch.from_numpy(desc1).to(device),
                    torch.from_numpy(desc2).to(device)
                )
                matches = matches['matches'].cpu().numpy().astype(np.uint32)
                
                # Сохранение совпадений в текстовый файл
                match_file = os.path.join(matches_dir, f"{image_name1}-{image_name2}.txt")
                with open(match_file, 'w') as mf:
                    for match in matches:
                        mf.write(f"{match[0]} {match[1]}\n")
                pf.write(f"{image_name1} {image_name2}\n")
    
    # Импорт совпадений в базу данных COLMAP
    try:
        result = subprocess.run([
            "colmap", "matches_importer",
            "--database_path", db_path,
            "--match_list_path", pairs_file,
            "--match_type", "pairs",
            "--import_path", matches_dir
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print("Matches importer output:", result.stdout)
        print("Matches importer errors:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error in matches_importer: {e.stderr}")
        shutil.rmtree(project_dir)
        return
    
    # Реконструкция сцены (взято из main.ipynb)
    os.makedirs(os.path.join(project_dir, "sparse"), exist_ok=True)
    try:
        result = subprocess.run([
            "colmap", "mapper",
            "--database_path", db_path,
            "--image_path", project_dir,
            "--output_path", os.path.join(project_dir, "sparse"),
            "--Mapper.min_num_matches", "3",
            "--Mapper.ignore_watermarks", "1",
            "--Mapper.init_min_num_inliers", "5"
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print("Mapper output:", result.stdout)
        print("Mapper errors:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error in mapper: {e.stderr}")
        shutil.rmtree(project_dir)
        return
    
    # Преобразование модели в текстовый формат
    try:
        result = subprocess.run([
            "colmap", "model_converter",
            "--input_path", os.path.join(project_dir, "sparse", "0"),
            "--output_path", os.path.join(project_dir, "sparse", "0"),
            "--output_type", "TXT"
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except subprocess.CalledProcessError as e:
        print(f"Error in model_converter: {e.stderr}")
        return
    
    # Парсинг images.txt для извлечения поз камер (взято из main.ipynb)
    images_txt_path = os.path.join(project_dir, "sparse", "0", "images.txt")
    if os.path.exists(images_txt_path):
        with open(images_txt_path, "r") as f:
            lines = f.readlines()
        image_data = {}
        i = 0
        while i < len(lines):
            if lines[i].startswith('#') or lines[i].strip() == '':
                i += 1
                continue
            parts = lines[i].strip().split()
            image_id, qw, qx, qy, qz, tx, ty, tz, camera_id, name = parts
            quat = [float(qw), float(qx), float(qy), float(qz)]
            trans = [float(tx), float(ty), float(tz)]
            rot_mat = R.from_quat(quat).as_matrix().flatten()
            rot_flat = ";".join(map(str, rot_mat))
            trans_flat = ";".join(map(str, trans))
            image_data[name] = (rot_flat, trans_flat)
            i += 2
    
        # Заполнение submission
        for idx in indices:
            row = group.loc[idx]
            image_name = row['image']
            if image_name in image_data:
                rot, trans = image_data[image_name]
                submission.at[idx, 'rotation_matrix'] = rot
                submission.at[idx, 'translation_vector'] = trans
            else:
                print(f"Warning: {image_name} not found in reconstruction.")
    
    # Удаление временной директории
    shutil.rmtree(project_dir)

In [ ]:
groups = submission.groupby(['dataset', 'scene'])

# Установка headless-режима для COLMAP (оставлено для совместимости, хотя не используется)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

In [1]:
for (dataset, scene), group in groups:
    dataset_path = os.path.join(test_path, dataset)
    if not os.path.exists(dataset_path):
        print(f"Skipping {dataset}")
        continue
    
    if dataset == 'ETs':
        image_groups = group_images_by_size(dataset, scene, group)
        for size_key, size_group in image_groups.items():
            print(f"Processing {size_key}")
            temp_group = pd.DataFrame([row for _, row in size_group])
            temp_indices = [idx for idx, _ in size_group]
            process_group(dataset, scene, temp_group, temp_indices)
    else:
        process_group(dataset, scene, group, group.index)

submission.to_csv('submission.csv', index=False)

NameError: name 'groups' is not defined